In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# 1. ĐỌC DỮ LIỆU
# ==========================================

DATA_DIR = Path("../../dataset")
INPUT_FILE = DATA_DIR / "tiki_products.csv"
OUTPUT_DIR = DATA_DIR / "cleaned"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

print("Kích thước dữ liệu ban đầu:", df.shape)


# ==========================================
# 2. CHUẨN HÓA TÊN CỘT
# ==========================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\-]+", "_", regex=True)
)

# Chuẩn hóa khoảng trắng ở các cột văn bản
text_cols = df.select_dtypes(include=["object"]).columns

for col in text_cols:
    df[col] = df[col].map(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Đã chuẩn hóa tên cột.")


# ==========================================
# 3. LOẠI BỎ CÁC CỘT KHÔNG HỮU ÍCH
# ==========================================

drop_cols = [
    "seller_name",       # Thiếu 100%
    "inventory_status",  # Thiếu 100%
    "order_count",       # Toàn bộ bằng 0
    "favourite_count",   # Toàn bộ bằng 0
    "has_installment",   # Toàn bộ False
    "has_gift",          # Toàn bộ False
    "availability"       # Toàn bộ bằng 1
]

# Chỉ xóa các cột thực sự tồn tại
drop_cols = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=drop_cols)

print("Các cột đã loại bỏ:", drop_cols)
print("Số cột còn lại:", len(df.columns))


# ==========================================
# 4. XỬ LÝ DỮ LIỆU TRÙNG LẶP
# ==========================================

before = len(df)

# Xóa bản ghi trùng hoàn toàn
df = df.drop_duplicates()

print("Số dòng trùng hoàn toàn đã xóa:", before - len(df))

# Kiểm tra ID trùng, không tự ý xóa sản phẩm
if "product_id" in df.columns:
    print(
        "Số product_id trùng:",
        df["product_id"].duplicated().sum()
    )

if "sku" in df.columns:
    print(
        "Số SKU trùng:",
        df["sku"].duplicated().sum()
    )


# ==========================================
# 5. CHUẨN HÓA KIỂU DỮ LIỆU
# ==========================================

numeric_cols = [
    "product_id",
    "sku",
    "seller_id",
    "price",
    "original_price",
    "discount",
    "discount_rate",
    "rating_average",
    "review_count",
    "quantity_sold",
    "category_id"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Chuyển thời gian crawl sang datetime
if "crawled_at" in df.columns:
    df["crawled_at"] = pd.to_datetime(
        df["crawled_at"], errors="coerce", utc=True
    )


# ==========================================
# 6. KIỂM TRA GIÁ VÀ TỶ LỆ GIẢM GIÁ
# ==========================================

# Tính lại mức giảm giá từ giá gốc và giá bán
valid_price = (
    df["price"].notna()
    & df["original_price"].notna()
    & (df["original_price"] > 0)
    & (df["price"] >= 0)
    & (df["price"] <= df["original_price"])
)

# Chỉ cập nhật các dòng có giá hợp lệ
df.loc[valid_price, "discount"] = (
    df.loc[valid_price, "original_price"]
    - df.loc[valid_price, "price"]
)

df.loc[valid_price, "discount_rate"] = (
    (
        df.loc[valid_price, "original_price"]
        - df.loc[valid_price, "price"]
    )
    / df.loc[valid_price, "original_price"]
    * 100
).round().astype(int)

# Không tự sửa các dòng có giá bất thường
print("Số dòng có giá không hợp lệ:", (~valid_price).sum())


# ==========================================
# 7. KIỂM TRA DỮ LIỆU ĐÁNH GIÁ
# ==========================================

if "rating_average" in df.columns:
    invalid_rating = ~df["rating_average"].between(
        0, 5, inclusive="both"
    )
    invalid_rating &= df["rating_average"].notna()

    print("Số rating ngoài phạm vi 0-5:", invalid_rating.sum())

if "review_count" in df.columns:
    invalid_review = df["review_count"] < 0
    print("Số review_count âm:", invalid_review.sum())


# ==========================================
# 8. XỬ LÝ GIÁ TRỊ THIẾU
# ==========================================

# Không tự điền 0 cho quantity_sold bị thiếu.
# Không xóa sản phẩm chỉ vì thiếu brand.
# Giữ NaN để tránh làm sai lệch dữ liệu.

print("\nSố giá trị thiếu sau xử lý:")
print(df.isna().sum()[df.isna().sum() > 0])
# ==========================================
# XÓA CỘT KHÔNG CẦN THIẾT
# ==========================================

drop_cols = ["sku", "inventory_status"]

df = df.drop(
    columns=[col for col in drop_cols if col in df.columns]
)

print("Đã xóa các cột:", drop_cols)


# ==========================================
# ĐỔI TÊN CỘT SANG TIẾNG VIỆT KHÔNG DẤU
# ==========================================

rename_cols = {
    "source": "nguon_du_lieu",
    "product_id": "ma_san_pham",
    "name": "ten_san_pham",
    "brand": "thuong_hieu",
    "seller_id": "ma_nguoi_ban",
    "seller_name": "ten_nguoi_ban",
    "price": "gia_ban",
    "original_price": "gia_goc",
    "discount": "so_tien_giam_gia",
    "discount_rate": "ty_le_giam_gia",
    "rating_average": "diem_danh_gia_trung_binh",
    "review_count": "so_luot_danh_gia",
    "quantity_sold": "so_luong_da_ban",
    "order_count": "so_luong_don_hang",
    "favourite_count": "so_luot_yeu_thich",
    "availability": "trang_thai_co_san",
    "is_authentic": "xac_thuc_chinh_hang",
    "has_installment": "co_ho_tro_tra_gop",
    "has_gift": "co_qua_tang",
    "category_id": "ma_danh_muc",
    "category_name": "ten_danh_muc",
    "category_path": "duong_dan_danh_muc",
    "url": "duong_dan_san_pham",
    "thumbnail_url": "duong_dan_anh_san_pham",
    "crawled_at": "thoi_gian_thu_thap"
}

df = df.rename(columns=rename_cols)

print("\nCác cột sau khi chuẩn hóa:")
print(df.columns.tolist())

# ==========================================
# 9. LƯU FILE ĐÃ LÀM SẠCH
# ==========================================

OUTPUT_FILE = OUTPUT_DIR / "tiki_cleaned.csv"

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nHOÀN TẤT LÀM SẠCH TIKI")
print("Kích thước cuối cùng:", df.shape)
print("File đã lưu tại:", OUTPUT_FILE)

Kích thước dữ liệu ban đầu: (3074, 27)
Đã chuẩn hóa tên cột.
Các cột đã loại bỏ: ['seller_name', 'inventory_status', 'order_count', 'favourite_count', 'has_installment', 'has_gift', 'availability']
Số cột còn lại: 20
Số dòng trùng hoàn toàn đã xóa: 0
Số product_id trùng: 0
Số SKU trùng: 0
Số dòng có giá không hợp lệ: 0
Số rating ngoài phạm vi 0-5: 0
Số review_count âm: 0

Số giá trị thiếu sau xử lý:
brand            496
quantity_sold    510
dtype: int64

HOÀN TẤT LÀM SẠCH TIKI
Kích thước cuối cùng: (3074, 20)
File đã lưu tại: ..\..\dataset\cleaned\tiki_cleaned.csv


C:\Users\LAPTOP\AppData\Local\Temp\ipykernel_27808\25644319.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include=["object"]).columns
